# Simple ZScore

The logic is:

High mvrv_zscore (overvalued, e.g. +3) → preference = −3 → buy less
Low mvrv_zscore (undervalued, e.g. −2) → preference = +2 → buy more
If mvrv_zscore column is missing (e.g. custom data without MVRV), falls back to np.zeros → uniform buying

#### Comparison to MVRV strategy:

| "" |SimpleZScoreStrategy|MVRVStrategy|
|---|---|---|
|Preference formula|−mvrv_zscore (one line)|Weighted blend of 3 signals (70/20/10)|
|Signals used|1 (mvrv_zscore)|3 (mvrv_zscore, mvrv_percentile, price_vs_ma)|
|Modulation|None|+ gradient, acceleration, volatility dampening|
|Purpose|Teaching example / baseline|Production strategy|

**example:**
&ensp;Aug 2, 2016 — price = $542.31, zscore = 0.1 <br>This day is somewhere in the middle of a 365-day window. Let's say it's day index d=180 (day 181 of 365).

**Step A — preference:**
preference = −0.1 <br>(zscore = 0.1 means Bitcoin is slightly overvalued -> mild signal to buy less)

**Step B — raw:**
$raw_{180} = \frac{1}{365} \exp{-0.1}$ = 0.002740×0.9048 = 0.002479 <br>For comparison, a perfectly neutral day (zscore = 0) would have:
$raw_{neutral} = \frac{1}{365} \exp{0}$ <br>= 0.002740

**Step C — stable_signal:**
Suppose the preceding 180 days had an average zscore of +0.8 (overvalued period). Then their raw values averaged around: <br>running_mean ≈ $\frac{1}{365} \exp{-0.8}$ ≈ 0.002740×0.449 = 0.001230
Then: <br>$signal_{180} = \frac{0.002479}{0.001230}$ ≈ 2.016 <br>Even though zscore = 0.1 means slightly overvalued, relative to the preceding very-overvalued days, this day looks like a buying opportunity.

**Step D — proposed:**
$proposed_{180} = 2.016 × \frac{1}{365}$ = 0.005523

**Step E — clipped:**
With 184 days remaining after this day, budget constraints give roughly: <br>max_upper = min(0.1, remaining − 184×0.00001) ≈ 0.1 <br>min_lower = max(0.00001, remaining − 184×0.1) <br>The proposed 0.005523 is within bounds → weight = 0.0055 (approximately)

In [17]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

from stacksats.model_development import precompute_features
from stacksats.runner import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.signals.simple_zscore import SimpleZScoreStrategy
from stacksats.strategies.stable.baselines.uniform import UniformStrategy

# Load raw BTC data
btc_df = pl.read_parquet("../data/raw/bitcoin_analytics.parquet").sort("date")
btc_df = btc_df.with_columns(pl.col("date").cast(pl.Datetime))

# Precompute MVRV z-score features for hover tooltips
features_df = precompute_features(btc_df).select(["date", "mvrv_zscore"])

# Strategy runner
runner = StrategyRunner()


def _dedup(batch, weight_col: str) -> pl.DataFrame:
    """Flatten overlapping rolling windows to one row per date.

    When a strategy runs over a multi-year range the runner produces
    overlapping rolling 365-day windows internally.  Each window assigns
    its own weight to every date it covers, so a single date may appear in
    dozens of windows with slightly different weights.  _dedup keeps the
    first weight seen for each date.
    """
    return (
        batch.to_dataframe()
        .group_by("date")
        .agg(
            pl.first("weight").alias(weight_col),
            pl.first("price_usd").alias("price_usd"),
        )
        .sort("date")
    )


In [10]:
# plotting only 2016 data
WINDOW_START = "2016-01-01"
WINDOW_END = "2016-12-31"
config = ExportConfig(range_start=WINDOW_START, range_end=WINDOW_END)

zscore_dd  = _dedup(runner.export(SimpleZScoreStrategy(), config, btc_df=btc_df), "zscore_weight")
uniform_dd = _dedup(runner.export(UniformStrategy(),       config, btc_df=btc_df), "uniform_weight").select(["date", "uniform_weight"])

merged = zscore_dd.join(uniform_dd, on="date", how="inner")
merged = merged.with_columns((1e8 / pl.col("price_usd")).alias("sats_per_dollar"))
merged = merged.join(features_df, on="date", how="left")

# Per-year heavy buy threshold (top 10% within this year only)
threshold = merged["zscore_weight"].quantile(0.90)
merged = merged.with_columns(
    (pl.col("zscore_weight") >= threshold).alias("is_heavy_buy")
)


# --- Interactive Plotting ---
fig = go.Figure()

# BTC Price (Left Axis - Log Scale)
fig.add_trace(go.Scatter(
    x=merged['date'], 
    y=merged['price_usd'],
    mode='lines',
    name='BTC Price (USD)',
    line=dict(color='black', width=1.5),
    yaxis='y1'
))

# Heavy Buy Markers (Left Axis)
heavy_buys = merged.filter(pl.col('is_heavy_buy'))
fig.add_trace(go.Scatter(
    x=heavy_buys['date'],
    y=heavy_buys['price_usd'],
    mode='markers',
    name='Heavy Buy Days',
    marker=dict(color='#16a34a', size=8, symbol='circle'),
    customdata=heavy_buys['zscore_weight'],
    hovertemplate="<b>Heavy Buy</b><br>Date: %{x}<br>Price: $%{y:,.2f}<br>Weight: %{customdata:.4f}<extra></extra>",
    yaxis='y1'
))

# Dynamic Allocation Weights (Right Axis - Bars)
fig.add_trace(go.Bar(
    x=merged['date'],
    y=merged['zscore_weight'],
    name='Dynamic Weight',
    marker_color='#16a34a',
    opacity=0.4,
    yaxis='y2',
    hovertemplate="Date: %{x}<br>Weight: %{y:.4f}<extra></extra>"
))

# Baseline Weight (Right Axis - Dashed Line)
fig.add_trace(go.Scatter(
    x=merged['date'],
    y=merged['uniform_weight'],
    mode='lines',
    name='Baseline Weight',
    line=dict(color='blue', dash='dash'),
    yaxis='y2'
))

# --- Layout and Styling ---
window_start = merged['date'].min().date()
window_end = merged['date'].max().date()

fig.update_layout(
    title=dict(
        text=(
            f"Dynamic DCA (Simple Z-Score) vs Baseline DCA<br>"
            f"<sup>{window_start} to {window_end} | "
            f"Dynamic SPD: {dynamic_spd:.0f} | Uniform SPD: {uniform_spd:.0f} | "
            f"Excess: {excess_pct:+.2f}%</sup>"
        ),
        x=0.5
    ),
    xaxis=dict(title="Date"),
    yaxis=dict(
        title="BTC Price (USD) - Log Scale",
        type="log",
        side="left",
        showgrid=True,
        tickprefix="$"
    ),
    yaxis2=dict(
        title="Allocation Weight (USD)",
        side="right",
        overlaying="y",
        showgrid=False
    ),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    template="plotly_white",
    hovermode="x unified",
    height=600
)

fig.show()

## 1. Full Window: 2010–2023

A single chart spanning the entire date range with Bitcoin halving cycle bands shaded in the background.  Heavy buy days (top 10% of SimpleZScore allocation weights globally) are marked as red circles.


In [11]:
WINDOW_START = "2010-01-01"
WINDOW_END   = "2023-12-31"
config = ExportConfig(range_start=WINDOW_START, range_end=WINDOW_END)

zscore_dd  = _dedup(runner.export(SimpleZScoreStrategy(), config, btc_df=btc_df), "zscore_weight")
uniform_dd = (
    _dedup(runner.export(UniformStrategy(), config, btc_df=btc_df), "uniform_weight")
    .select(["date", "uniform_weight"])
)

merged_full = zscore_dd.join(uniform_dd, on="date", how="inner")
merged_full = merged_full.with_columns((1e8 / pl.col("price_usd")).alias("sats_per_dollar"))

zscore_spd  = (merged_full["zscore_weight"]  * merged_full["sats_per_dollar"]).sum() / merged_full["zscore_weight"].sum()
uniform_spd = (merged_full["uniform_weight"] * merged_full["sats_per_dollar"]).sum() / merged_full["uniform_weight"].sum()
excess_pct  = (zscore_spd - uniform_spd) / uniform_spd * 100

threshold = merged_full["zscore_weight"].quantile(0.90)
merged_full = merged_full.join(features_df, on="date", how="left").with_columns(
    (pl.col("zscore_weight") >= threshold).alias("heavy_buy")
)
df = merged_full.to_pandas()

HALVING_CYCLES = [
    ("Pre-2012 Halving",        "2010-01-01", "2012-11-28", "rgba(173,216,230,0.25)"),
    ("2012-2016 Halving Cycle", "2012-11-28", "2016-07-09", "rgba(144,238,144,0.25)"),
    ("2016-2020 Halving Cycle", "2016-07-09", "2020-05-11", "rgba(173,216,230,0.25)"),
    ("2020-2024 Halving Cycle", "2020-05-11", "2023-12-31", "rgba(144,238,144,0.25)"),
]
HALVING_DATES = ["2012-11-28", "2016-07-09", "2020-05-11"]

fig = make_subplots(specs=[[{"secondary_y": True}]])

for label, c_start, c_end, fillcolor in HALVING_CYCLES:
    fig.add_vrect(
        x0=c_start, x1=c_end,
        fillcolor=fillcolor, layer="below", line_width=0,
        annotation_text=label,
        annotation_position="top left",
        annotation=dict(font=dict(size=11, color="#475569")),
    )

for halving_date in HALVING_DATES:
    fig.add_vline(x=halving_date, line_dash="dash", line_color="#94a3b8", line_width=1.2)

# BTC price line
fig.add_trace(
    go.Scatter(
        x=df["date"], y=df["price_usd"],
        name="BTC Price (USD)",
        line=dict(color="#334155", width=1.8),
    ),
    secondary_y=False,
)

# Heavy buy markers
heavy = df[df["heavy_buy"]]
fig.add_trace(
    go.Scatter(
        x=heavy["date"], y=heavy["price_usd"],
        mode="markers", name="Heavy Buy Day",
        marker=dict(color="red", size=8, line=dict(color="black", width=1)),
        customdata=heavy[["zscore_weight", "mvrv_zscore"]].values,
        hovertemplate="Weight: %{customdata[0]:.7f}<br>MVRV Z: %{customdata[1]:.2f}<extra></extra>",
    ),
    secondary_y=False,
)

# SimpleZScore allocation weight (area fill)
fig.add_trace(
    go.Scatter(
        x=df["date"], y=df["zscore_weight"],
        name="SimpleZScore Weight",
        line=dict(color="green", width=1.5),
        fill="tozeroy", fillcolor="rgba(0,128,0,0.35)",
        hovertemplate="Weight: %{y:.7f}<extra></extra>",
    ),
    secondary_y=True,
)

# Baseline (Uniform) weight
fig.add_trace(
    go.Scatter(
        x=df["date"], y=df["uniform_weight"],
        name="Baseline Weight",
        line=dict(color="blue", dash="dash", width=1.5),
    ),
    secondary_y=True,
)

fig.update_layout(
    title=dict(
        text=(
            f"SimpleZScore vs Uniform Strategy {WINDOW_START} to {WINDOW_END}<br>"
            f"<sup>SimpleZScore SPD: {zscore_spd:,.0f} | Uniform SPD: {uniform_spd:,.0f} | "
            f"Excess: {excess_pct:+.2f}%</sup>"
        ),
        x=0.5, font=dict(size=18),
    ),
    hovermode="x unified",
    xaxis=dict(title="Year", showgrid=True),
    yaxis=dict(title="BTC Price (USD) — Log Scale", type="log", tickprefix="$", showgrid=True),
    yaxis2=dict(title="Allocation Weight", showgrid=False),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    height=700,
    template="plotly_white",
)

fig.show()


## 2. Year-by-Year Analysis: 2010–2023

Each year is run as an independent 365-day strategy window.  Heavy buy days
(top 10% of weights within each year) are marked as red circles.

The SPD summary table is displayed below the chart.


In [12]:
YEARS = range(2010, 2024)
year_data = []  # list of (year, pandas DataFrame)

for year in YEARS:
    yr_start = f"{year}-01-01"
    yr_end   = f"{year}-12-31"
    config   = ExportConfig(range_start=yr_start, range_end=yr_end)

    zscore_dd  = _dedup(runner.export(SimpleZScoreStrategy(), config, btc_df=btc_df), "zscore_weight")
    uniform_dd = (
        _dedup(runner.export(UniformStrategy(), config, btc_df=btc_df), "uniform_weight")
        .select(["date", "uniform_weight"])
    )

    merged_yr = zscore_dd.join(uniform_dd, on="date", how="inner")
    merged_yr = merged_yr.with_columns((1e8 / pl.col("price_usd")).alias("sats_per_dollar"))
    merged_yr = merged_yr.join(features_df, on="date", how="left")

    threshold = merged_yr["zscore_weight"].quantile(0.90)
    merged_yr = merged_yr.with_columns(
        (pl.col("zscore_weight") >= threshold).alias("heavy_buy")
    )
    year_data.append((year, merged_yr.to_pandas()))

# Build faceted figure
spd_rows = []
n_years  = len(year_data)

fig = make_subplots(
    rows=n_years, cols=1,
    specs=[[{"secondary_y": True}] for _ in range(n_years)],
    shared_xaxes=False,
    vertical_spacing=0.02,
    subplot_titles=[str(y) for y, _ in year_data],
)

for i, (year, yr_df) in enumerate(year_data, start=1):
    heavy     = yr_df[yr_df["heavy_buy"]]
    yz_spd    = (yr_df["zscore_weight"]  * yr_df["sats_per_dollar"]).sum() / yr_df["zscore_weight"].sum()
    yu_spd    = (yr_df["uniform_weight"] * yr_df["sats_per_dollar"]).sum() / yr_df["uniform_weight"].sum()
    yr_excess = (yz_spd - yu_spd) / yu_spd * 100

    spd_rows.append({
        "year":        year,
        "uniform_spd": round(float(yu_spd)),
        "zscore_spd":  round(float(yz_spd)),
        "excess_pct":  round(float(yr_excess), 2),
    })

    fig.layout.annotations[i - 1].update(
        text=(
            f"<b>{year}</b>  |  ZScore SPD: {yz_spd:,.0f}  "
            f"Uniform SPD: {yu_spd:,.0f}  Excess: {yr_excess:+.2f}%"
        ),
        font=dict(size=11),
    )

    show_legend = (i == 1)

    # SimpleZScore weight bars (right axis)
    fig.add_trace(
        go.Bar(
            x=yr_df["date"], y=yr_df["zscore_weight"],
            name="SimpleZScore Weight",
            marker_color="green", opacity=0.3,
            hovertemplate="Weight: %{y:.7f}<extra></extra>",
            showlegend=show_legend, legendgroup="zscore",
        ),
        row=i, col=1, secondary_y=True,
    )

    # Baseline weight line (right axis)
    fig.add_trace(
        go.Scatter(
            x=yr_df["date"], y=yr_df["uniform_weight"],
            name="Baseline Weight",
            line=dict(color="blue", dash="dash", width=1.5),
            showlegend=show_legend, legendgroup="baseline",
        ),
        row=i, col=1, secondary_y=True,
    )

    # BTC price line (left axis)
    fig.add_trace(
        go.Scatter(
            x=yr_df["date"], y=yr_df["price_usd"],
            name="BTC Price (USD)",
            line=dict(color="#334155", width=1.5),
            showlegend=show_legend, legendgroup="price",
        ),
        row=i, col=1, secondary_y=False,
    )

    # Heavy buy markers (left axis)
    fig.add_trace(
        go.Scatter(
            x=heavy["date"], y=heavy["price_usd"],
            mode="markers", name="Heavy Buy Day",
            marker=dict(color="red", size=7, line=dict(color="white", width=1)),
            customdata=heavy[["zscore_weight", "mvrv_zscore"]].values,
            hovertemplate="Weight: %{customdata[0]:.7f}<br>MVRV Z: %{customdata[1]:.2f}<extra></extra>",
            showlegend=show_legend, legendgroup="heavy",
        ),
        row=i, col=1, secondary_y=False,
    )

    fig.update_yaxes(
        title_text="BTC Price Log-Scale", type="log", tickprefix="$",
        showgrid=True, gridcolor="#f1f5f9", row=i, col=1, secondary_y=False,
    )
    fig.update_yaxes(title_text="Allocation Weight", showgrid=False, row=i, col=1, secondary_y=True)
    fig.update_xaxes(showgrid=True, gridcolor="#f1f5f9", row=i, col=1)

fig.update_layout(
    title=dict(
        text="SimpleZScore vs Uniform Strategy — 2010 to 2023 (by Year)",
        x=0.5, font=dict(size=18),
    ),
    template="plotly_white",
    hovermode="x unified",
    height=400 * n_years,
    legend=dict(orientation="h", yanchor="bottom", y=1.002, xanchor="right", x=1),
)

fig.show()

# SPD Summary Table
spd_summary = pl.DataFrame(spd_rows).rename({
    "year":        "Year",
    "uniform_spd": "Uniform SPD (sats/$)",
    "zscore_spd":  "SimpleZScore SPD (sats/$)",
    "excess_pct":  "SimpleZScore vs Uniform Excess (%)",
})
spd_summary


Year,Uniform SPD (sats/$),SimpleZScore SPD (sats/$),SimpleZScore vs Uniform Excess (%)
i64,i64,i64,f64
2010,362078691,362078691,0.0
2011,63364681,63964909,0.95
2012,14015557,12012604,-14.29
2013,1583978,1513959,-4.42
2014,204888,191598,-6.49
…,…,…,…
2019,15921,16467,3.43
2020,10038,9521,-5.15
2021,2206,2300,4.28


## Plots similar to Momentum strategy

In [19]:
calendar_cycles = [
    {
        "label": "2010-2013",
        "start": "2010-08-16",
        "end": "2013-12-31"
    },
    {
        "label": "2014-2017",
        "start": "2014-01-01",
        "end": "2017-12-31"
    },
    {
        "label": "2018-2021",
        "start": "2018-01-01",
        "end": "2021-12-31"
    },
    {
        "label": "2022-2023",
        "start": "2022-01-01",
        "end": "2023-12-31"
    }
]

halving_periods = [
    {
        "label": "Pre-2012 Halving",
        "start": "2010-08-16",
        "end": "2012-11-27",
        "color": "rgba(173, 216, 230, 0.10)"
    },
    {
        "label": "2012-2016 Halving Cycle",
        "start": "2012-11-28",
        "end": "2016-07-08",
        "color": "rgba(144, 238, 144, 0.10)"
    },
    {
        "label": "2016-2020 Halving Cycle",
        "start": "2016-07-09",
        "end": "2020-05-10",
        "color": "rgba(221, 160, 221, 0.10)"
    },
    {
        "label": "2020-2024 Halving Cycle",
        "start": "2020-05-11",
        "end": "2023-12-31",
        "color": "rgba(255, 165, 0, 0.10)"
    }
]

def export_one_year(
    strategy,
    btc_data: pl.DataFrame,
    year: int,
    runner: StrategyRunner
) -> pl.DataFrame | None:
    
    year_start = f"{year}-01-01"
    year_end = f"{year}-12-31"

    year_df = (
        btc_data
        .filter(
            (pl.col("date") >= pl.datetime(year, 1, 1)) &
            (pl.col("date") <= pl.datetime(year, 12, 31))
        )
        .sort("date")
    )

    if year_df.is_empty():
        print(f"{year}: skipped, no data")
        return None

    if year_df.height < 365:
        print(f"{year}: skipped, less than 365 rows")
        return None

    try:
        config = ExportConfig(
            range_start=year_start,
            range_end=year_end
        )

        export_obj = runner.export(strategy, config, btc_df=year_df)
        df = export_obj.to_dataframe()

        df = df.with_columns([
            pl.col("start_date").cast(pl.Datetime),
            pl.col("end_date").cast(pl.Datetime),
            pl.col("date").cast(pl.Datetime),
        ])

        # For leap years, export may return two 365-day windows.
        # Keep the one ending latest.
        latest_end = df.select(pl.col("end_date").max()).item()

        df_one = (
            df
            .filter(pl.col("end_date") == latest_end)
            .sort("date")
        )

        print(f"{year}: exported {df_one.height} rows")
        return df_one

    except Exception as e:
        print(f"{year}: skipped due to error -> {e}")
        return None
    
def process_cycle_year_by_year(
    cycle: dict,
    btc_data: pl.DataFrame,
    runner: StrategyRunner,
    total_budget_usd: float = 1000.0,
    top_buy_quantile: float = 0.90
):
    cycle_label = cycle["label"]
    cycle_start = cycle["start"]
    cycle_end = cycle["end"]

    start_year = int(cycle_start[:4])
    end_year = int(cycle_end[:4])

    print(f"\nProcessing {cycle_label}")

    cycle_df = (
        btc_data
        .filter(
            (pl.col("date") >= pl.lit(cycle_start).str.to_datetime()) &
            (pl.col("date") <= pl.lit(cycle_end).str.to_datetime())
        )
        .sort("date")
    )

    print(
        cycle_df.select(
            pl.col("date").min().alias("min_date"),
            pl.col("date").max().alias("max_date"),
            pl.len().alias("rows")
        )
    )

    simple_z_score_strategy = SimpleZScoreStrategy()
    uniform_strategy = UniformStrategy()

    simple_z_score_yearly = []
    uniform_yearly = []

    for year in range(start_year, end_year + 1):
        simple_z_score_result = export_one_year(simple_z_score_strategy, cycle_df, year, runner)
        if simple_z_score_result is not None:
            simple_z_score_yearly.append(simple_z_score_result)

        uniform_result = export_one_year(uniform_strategy, cycle_df, year, runner)
        if uniform_result is not None:
            uniform_yearly.append(uniform_result)

    if not simple_z_score_yearly:
        raise ValueError(f"No valid SimpleZScore exports for {cycle_label}")

    if not uniform_yearly:
        raise ValueError(f"No valid Uniform exports for {cycle_label}")

    simple_z_score_all = pl.concat(simple_z_score_yearly).rename({"weight": "simple_z_score_weight_raw"})
    uniform_all = pl.concat(uniform_yearly).rename({"weight": "baseline_weight_raw"})

    merged = (
        simple_z_score_all
        .select(["date", "price_usd", "simple_z_score_weight_raw"])
        .join(
            uniform_all.select(["date", "baseline_weight_raw"]),
            on="date",
            how="inner"
        )
        .sort("date")
    )

    # Normalize both strategies inside this cycle.
    simple_z_score_sum = merged["simple_z_score_weight_raw"].sum()
    baseline_sum = merged["baseline_weight_raw"].sum()

    merged = merged.with_columns([
        (pl.col("simple_z_score_weight_raw") / simple_z_score_sum).alias("simple_z_score_weight"),
        (pl.col("baseline_weight_raw") / baseline_sum).alias("baseline_weight"),
    ])

    # Convert weights into USD allocation.
    merged = merged.with_columns([
        (pl.col("simple_z_score_weight") * total_budget_usd).alias("dynamic_usd"),
        (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
    ])

    # Convert USD allocation into BTC.
    merged = merged.with_columns([
        (pl.col("dynamic_usd") / pl.col("price_usd")).alias("btc_accum_dynamic"),
        (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
    ])

    # Convert BTC into sats.
    merged = merged.with_columns([
        (pl.col("btc_accum_dynamic") * 100_000_000).alias("sats_accum_dynamic"),
        (pl.col("btc_accum_baseline") * 100_000_000).alias("sats_accum_baseline"),
    ])

    # Daily sats per dollar.
    merged = merged.with_columns([
        (pl.col("sats_accum_dynamic") / pl.col("dynamic_usd")).alias("sats_per_dollar_dynamic"),
        (pl.col("sats_accum_baseline") / pl.col("baseline_usd")).alias("sats_per_dollar_baseline"),
    ])

    total_dynamic_btc = merged["btc_accum_dynamic"].sum()
    total_baseline_btc = merged["btc_accum_baseline"].sum()

    sats_per_dollar_dynamic = (total_dynamic_btc / total_budget_usd) * 100_000_000
    sats_per_dollar_baseline = (total_baseline_btc / total_budget_usd) * 100_000_000

    pct_diff_vs_baseline = (
        (total_dynamic_btc - total_baseline_btc) / total_baseline_btc
    ) * 100

    performance_label = "better" if pct_diff_vs_baseline > 0 else "worse"

    # Top buy days = top 10% of Simple Z-Score weights within this cycle.
    top_buy_threshold = merged["simple_z_score_weight"].quantile(top_buy_quantile)

    top_buy_points = (
        merged
        .filter(pl.col("simple_z_score_weight") >= top_buy_threshold)
        .select(["date", "price_usd", "simple_z_score_weight"])
        .sort("simple_z_score_weight", descending=True)
        .to_pandas()
    )

    top_buy_points["date"] = pd.to_datetime(top_buy_points["date"])

    plot_df = merged.to_pandas()
    plot_df["date"] = pd.to_datetime(plot_df["date"])

    return {
        "label": cycle_label,
        "start": cycle_start,
        "end": cycle_end,
        "merged": merged,
        "plot_df": plot_df,
        "top_buy_points": top_buy_points,
        "top_buy_threshold": top_buy_threshold,
        "top_buy_quantile": top_buy_quantile,
        "total_dynamic_btc": total_dynamic_btc,
        "total_baseline_btc": total_baseline_btc,
        "sats_per_dollar_dynamic": sats_per_dollar_dynamic,
        "sats_per_dollar_baseline": sats_per_dollar_baseline,
        "pct_diff_vs_baseline": pct_diff_vs_baseline,
        "performance_label": performance_label,
    }

cycle_results = {}

for cycle in calendar_cycles:
    result = process_cycle_year_by_year(
        cycle=cycle,
        btc_data=btc_df,
        runner=runner,
        total_budget_usd=1000.0,
        top_buy_quantile=0.90
    )

    cycle_results[cycle["label"]] = result

# Create one interactive Plotly chart for one cycle.
# This uses the year-by-year export result, so it should match the static chart methodology.
import plotly.graph_objects as go
def plot_interactive_cycle_yearly(result: dict, halving_periods: list[dict]):
    plot_df = result["plot_df"].copy()
    top_buy_points = result["top_buy_points"].copy()

    cycle_start = pd.to_datetime(result["start"])
    cycle_end = pd.to_datetime(result["end"])

    fig = go.Figure()

    # Actual halving-period background shading, clipped to this cycle.
    for period in halving_periods:
        period_start = pd.to_datetime(period["start"])
        period_end = pd.to_datetime(period["end"])

        shaded_start = max(period_start, cycle_start)
        shaded_end = min(period_end, cycle_end)

        if shaded_start <= shaded_end:
            fig.add_vrect(
                x0=shaded_start,
                x1=shaded_end,
                fillcolor=period["color"],
                opacity=1.0,
                layer="below",
                line_width=0,
                annotation_text=period["label"],
                annotation_position="top left",
                annotation_font_size=11,
            )

    # BTC price line.
    fig.add_trace(
        go.Scatter(
            x=plot_df["date"],
            y=plot_df["price_usd"],
            mode="lines",
            name="BTC Price (USD)",
            line=dict(color="black", width=2),
            yaxis="y1",
            customdata=plot_df[[
                "simple_z_score_weight",
                "baseline_weight",
                "sats_per_dollar_dynamic",
                "sats_per_dollar_baseline",
                "sats_accum_dynamic",
                "sats_accum_baseline",
            ]],
            hovertemplate=(
                "<b>Date</b>: %{x|%Y-%m-%d}<br>"
                "<b>BTC Price</b>: $%{y:,.2f}<br>"
                "<b>Simple Z-Score Weight</b>: %{customdata[0]:.8f}<br>"
                "<b>Uniform  Weight</b>: %{customdata[1]:.8f}<br>"
                "<b>Simple Z-Score sats/$</b>: %{customdata[2]:,.2f}<br>"
                "<b>Uniform sats/$</b>: %{customdata[3]:,.2f}<br>"
                "<b>Simple Z-Score sats accumulated</b>: %{customdata[4]:,.2f}<br>"
                "<b>Uniform sats accumulated</b>: %{customdata[5]:,.2f}"
                "<extra></extra>"
            )
        )
    )

    # Simple Z-Score allocation area.
    fig.add_trace(
        go.Scatter(
            x=plot_df["date"],
            y=plot_df["simple_z_score_weight"],
            mode="lines",
            name="Simple Z-Score Weight",
            line=dict(color="green", width=1.5),
            fill="tozeroy",
            fillcolor="green", opacity=0.3,
            yaxis="y2",
            customdata=plot_df[[
                "price_usd",
                "sats_per_dollar_dynamic",
                "sats_accum_dynamic",
            ]],
            hovertemplate=(
                "<b>Date</b>: %{x|%Y-%m-%d}<br>"
                "<b>Simple Z-Score Weight</b>: %{y:.8f}<br>"
                "<b>BTC Price</b>: $%{customdata[0]:,.2f}<br>"
                "<b>Simple Z-Score sats/$</b>: %{customdata[1]:,.2f}<br>"
                "<b>Simple Z-Score sats accumulated</b>: %{customdata[2]:,.2f}"
                "<extra></extra>"
            )
        )
    )

    # Baseline DCA line.
    fig.add_trace(
        go.Scatter(
            x=plot_df["date"],
            y=plot_df["baseline_weight"],
            mode="lines",
            name="Baseline Weight",
            line=dict(color="blue", width=2, dash="dash"),
            yaxis="y2",
            customdata=plot_df[[
                "price_usd",
                "sats_per_dollar_baseline",
                "sats_accum_baseline",
            ]],
            hovertemplate=(
                "<b>Date</b>: %{x|%Y-%m-%d}<br>"
                "<b>Baseline Weight</b>: %{y:.8f}<br>"
                "<b>BTC Price</b>: $%{customdata[0]:,.2f}<br>"
                "<b>Baseline sats/$</b>: %{customdata[1]:,.2f}<br>"
                "<b>Baseline sats accumulated</b>: %{customdata[2]:,.2f}"
                "<extra></extra>"
            )
        )
    )

    # Top 10% buy 
    fig.add_trace(
        go.Scatter(
            x=top_buy_points["date"],
            y=top_buy_points["price_usd"],
            mode="markers",
            name="Top 10% Buy Days",
            marker=dict(
                color="red",
                line=dict(color="black", width=1)
            ),
            yaxis="y1",
            customdata=top_buy_points[["simple_z_score_weight"]],
            hovertemplate=(
                "<b>Top 10% Buy Date</b>: %{x|%Y-%m-%d}<br>"
                "<b>BTC Price</b>: $%{y:,.2f}<br>"
                "<b>Simple Z-Score Weight</b>: %{customdata[0]:.8f}"
                "<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=(
            f"Simple Z-Score Strategy vs Uniform Strategy - {result['label']}<br>"
            f"ZScore SPD: {result['sats_per_dollar_dynamic']:.2f} | "
            f"Uniform SPD: {result['sats_per_dollar_baseline']:.2f} | "
            f"Excess: {result['pct_diff_vs_baseline']:.2f}% | "
            f"Top buy threshold: {result['top_buy_threshold']:.8f}"
        ),
        width=1250,
        height=650,
        hovermode="x unified",
        xaxis=dict(
            title="Date",
            range=[cycle_start, cycle_end]
        ),
        yaxis=dict(
            title="BTC Price (USD, log scale)",
            type="log",
            side="left"
        ),
        yaxis2=dict(
            title="Allocation Weight",
            overlaying="y",
            side="right",
            showgrid=False
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="left",
            x=0
        ),
        margin=dict(l=70, r=70, t=120, b=60)
    )

    fig.show()


# Create 4 separate interactive charts, one for each cycle.

for label, result in cycle_results.items():
    plot_interactive_cycle_yearly(result, halving_periods)


Processing 2010-2013
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2010-08-16 00:00:00 ┆ 2013-12-31 00:00:00 ┆ 1234 │
└─────────────────────┴─────────────────────┴──────┘
2010: skipped, less than 365 rows
2010: skipped, less than 365 rows
2011: exported 365 rows
2011: exported 365 rows
2012: exported 365 rows
2012: exported 365 rows
2013: exported 365 rows
2013: exported 365 rows

Processing 2014-2017
shape: (1, 3)
┌─────────────────────┬─────────────────────┬──────┐
│ min_date            ┆ max_date            ┆ rows │
│ ---                 ┆ ---                 ┆ ---  │
│ datetime[μs]        ┆ datetime[μs]        ┆ u32  │
╞═════════════════════╪═════════════════════╪══════╡
│ 2014-01-01 00:00:00 ┆ 2017-12-31 00:00:00 ┆ 1461 │
└─────────────────────┴────

In [20]:
# Create one combined interactive Plotly chart across all 4 cycles.
# This uses the same year-by-year export results stored in cycle_results.

import plotly.graph_objects as go
import pandas as pd
import polars as pl

# Combine all merged cycle dataframes into one dataframe
combined_merged = pl.concat(
    [result["merged"] for result in cycle_results.values()]
).sort("date")

# Convert to pandas for plotting
combined_plot_df = combined_merged.to_pandas()
combined_plot_df["date"] = pd.to_datetime(combined_plot_df["date"])
combined_plot_df = combined_plot_df.sort_values("date")

# Add cycle label for tooltip
def assign_cycle_label(date):
    for cycle in calendar_cycles:
        start = pd.to_datetime(cycle["start"])
        end = pd.to_datetime(cycle["end"])
        if start <= date <= end:
            return cycle["label"]
    return "Outside cycle"

combined_plot_df["cycle_label"] = combined_plot_df["date"].apply(assign_cycle_label)

# Top buy dates across all cycles combined
top_buy_points_combined = (
    combined_merged
    .sort("simple_z_score_weight", descending=True)
    .head(25)
    .select(["date", "price_usd", "simple_z_score_weight"])
    .to_pandas()
)

top_buy_points_combined["date"] = pd.to_datetime(top_buy_points_combined["date"])

# Overall totals across all cycles
total_dynamic_btc_combined = combined_merged["btc_accum_dynamic"].sum()
total_baseline_btc_combined = combined_merged["btc_accum_baseline"].sum()

# Each cycle used 1000 USD in your earlier logic
total_budget_usd_combined = 1000.0 * len(cycle_results)

sats_per_dollar_dynamic_combined = (
    total_dynamic_btc_combined / total_budget_usd_combined
) * 100_000_000

sats_per_dollar_baseline_combined = (
    total_baseline_btc_combined / total_budget_usd_combined
) * 100_000_000

pct_diff_vs_baseline_combined = (
    (total_dynamic_btc_combined - total_baseline_btc_combined)
    / total_baseline_btc_combined
) * 100

performance_label_combined = (
    "better" if pct_diff_vs_baseline_combined > 0 else "worse"
)

# Create figure
fig = go.Figure()

# Add actual halving-period background shading
for period in halving_periods:
    fig.add_vrect(
        x0=pd.to_datetime(period["start"]),
        x1=pd.to_datetime(period["end"]),
        fillcolor=period["color"],
        opacity=1.0,
        layer="below",
        line_width=0,
        annotation_text=period["label"],
        annotation_position="top left",
        annotation_font_size=11,
    )

# Add vertical lines for cycle boundaries
for cycle in calendar_cycles:
    fig.add_vline(
        x=pd.to_datetime(cycle["start"]),
        line_width=1,
        line_dash="dot",
        line_color="gray"
    )

# BTC price trace on left y-axis (log scale)
fig.add_trace(
    go.Scatter(
        x=combined_plot_df["date"],
        y=combined_plot_df["price_usd"],
        mode="lines",
        name="BTC Price (USD)",
        line=dict(color="black", width=2),
        yaxis="y1",
        customdata=combined_plot_df[
            [
                "cycle_label",
                "simple_z_score_weight",
                "baseline_weight",
                "sats_per_dollar_dynamic",
                "sats_per_dollar_baseline",
                "sats_accum_dynamic",
                "sats_accum_baseline",
            ]
        ],
        hovertemplate=(
            "<b>Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>Cycle</b>: %{customdata[0]}<br>"
            "<b>BTC Price</b>: $%{y:,.2f}<br>"
            "<b>Simple Z-Score Weight</b>: %{customdata[1]:.8f}<br>"
            "<b>DCA Weight</b>: %{customdata[2]:.8f}<br>"
            "<b>Simple Z-Score sats/$</b>: %{customdata[3]:,.2f}<br>"
            "<b>DCA sats/$</b>: %{customdata[4]:,.2f}<br>"
            "<b>Simple Z-Score sats accumulated</b>: %{customdata[5]:,.2f}<br>"
            "<b>DCA sats accumulated</b>: %{customdata[6]:,.2f}"
            "<extra></extra>"
        ),
    )
)

# Simple Z-Score weight on right y-axis
fig.add_trace(
    go.Scatter(
        x=combined_plot_df["date"],
        y=combined_plot_df["simple_z_score_weight"],
        mode="lines",
        name="Simple Z-Score Weight",
        line=dict(color="green", width=1.5),
        fill="tozeroy",
        fillcolor="rgba(0, 128, 0, 0.35)",
        yaxis="y2",
        customdata=combined_plot_df[
            [
                "cycle_label",
                "price_usd",
                "sats_per_dollar_dynamic",
                "sats_accum_dynamic",
            ]
        ],
        hovertemplate=(
            "<b>Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>Cycle</b>: %{customdata[0]}<br>"
            "<b>Simple Z-Score Weight</b>: %{y:.8f}<br>"
            "<b>BTC Price</b>: $%{customdata[1]:,.2f}<br>"
            "<b>Simple Z-Score sats/$</b>: %{customdata[2]:,.2f}<br>"
            "<b>Simple Z-Score sats accumulated</b>: %{customdata[3]:,.2f}"
            "<extra></extra>"
        ),
    )
)

# Baseline DCA on right y-axis
fig.add_trace(
    go.Scatter(
        x=combined_plot_df["date"],
        y=combined_plot_df["baseline_weight"],
        mode="lines",
        name="Baseline DCA Weight",
        line=dict(color="orange", width=2, dash="dash"),
        yaxis="y2",
        customdata=combined_plot_df[
            [
                "cycle_label",
                "price_usd",
                "sats_per_dollar_baseline",
                "sats_accum_baseline",
            ]
        ],
        hovertemplate=(
            "<b>Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>Cycle</b>: %{customdata[0]}<br>"
            "<b>DCA Weight</b>: %{y:.8f}<br>"
            "<b>BTC Price</b>: $%{customdata[1]:,.2f}<br>"
            "<b>DCA sats/$</b>: %{customdata[2]:,.2f}<br>"
            "<b>DCA sats accumulated</b>: %{customdata[3]:,.2f}"
            "<extra></extra>"
        ),
    )
)

# Top buy dates
fig.add_trace(
    go.Scatter(
        x=top_buy_points_combined["date"],
        y=top_buy_points_combined["price_usd"],
        mode="markers",
        name="Top Buy Dates",
        marker=dict(
            color="green",
            symbol="triangle-up",
            size=11,
            line=dict(color="black", width=1),
        ),
        yaxis="y1",
        customdata=top_buy_points_combined[["simple_z_score_weight"]],
        hovertemplate=(
            "<b>Top Buy Date</b>: %{x|%Y-%m-%d}<br>"
            "<b>BTC Price</b>: $%{y:,.2f}<br>"
            "<b>Simple Z-Score Weight</b>: %{customdata[0]:.8f}"
            "<extra></extra>"
        ),
    )
)

# Final layout
fig.update_layout(
    title=(
        "Simple Z-Score Strategy vs Baseline DCA Across All 4 Cycles<br>"
        f"Dynamic BTC: {total_dynamic_btc_combined:.6f} | "
        f"DCA BTC: {total_baseline_btc_combined:.6f} | "
        f"Simple Z-Score performed {abs(pct_diff_vs_baseline_combined):.2f}% "
        f"{performance_label_combined} than DCA"
    ),
    width=1400,
    height=750,
    hovermode="x unified",
    xaxis=dict(
        title="Date",
        range=[
            pd.to_datetime("2010-08-16"),
            pd.to_datetime("2023-12-31")
        ],
    ),
    yaxis=dict(
        title="BTC Price (USD, log scale)",
        type="log",
        side="left",
    ),
    yaxis2=dict(
        title="Allocation Weight",
        overlaying="y",
        side="right",
        showgrid=False,
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0,
    ),
    margin=dict(l=70, r=70, t=120, b=60),
)

fig.show()